# HBE_2206 | ML-00 — Machine Learning Fundamentals & Workflow
**Author:** HAZEM ELSHAFEY  
**Track:** Python for AI → Machine Learning  
**Status:** Starter learning notebook — validate and extend during review.

## Learning objectives
- Distinguish features from the target.
- Split data into training and test sets.
- Compare a baseline with a simple regression model.
- Evaluate predictions with MAE and RMSE.
- Explain why test data must not guide model selection.

This notebook uses synthetic data for teaching the workflow. It does **not** establish real-world predictive performance.

## 1. Imports and reproducibility
We set a random seed so the generated example can be reproduced.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print('Imports complete. Random state:', RANDOM_STATE)

**Expected output**
```text
Imports complete. Random state: 42
```

## 2. Create a small synthetic dataset
The target is generated from two features plus random noise. This controlled setup helps us focus on the workflow.

In [ ]:
n_samples = 200
hours_studied = rng.uniform(0, 10, size=n_samples)
practice_tasks = rng.uniform(0, 20, size=n_samples)
noise = rng.normal(0, 3, size=n_samples)
score = 10 + 4.0 * hours_studied + 1.5 * practice_tasks + noise

df = pd.DataFrame({
    'hours_studied': hours_studied,
    'practice_tasks': practice_tasks,
    'score': score,
})
df.head()

**Expected output:** a five-row DataFrame with columns `hours_studied`, `practice_tasks`, and `score`. Numeric values are reproducible for the same NumPy version and random seed.

## 3. Identify features and target
Features (`X`) are inputs used for prediction. The target (`y`) is the value we want to predict. Do not include the target among the input features.

In [ ]:
X = df[['hours_studied', 'practice_tasks']]
y = df['score']

print('Feature columns:', X.columns.tolist())
print('Number of rows:', len(df))
print('Target name:', y.name)

**Expected output**
```text
Feature columns: ['hours_studied', 'practice_tasks']
Number of rows: 200
Target name: score
```

## 4. Split the data
The test set is held back for evaluation. In real workflows, all learned preprocessing must be fitted using training data only; scikit-learn Pipelines help enforce this for more complex preprocessing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print('Training rows:', len(X_train))
print('Test rows:', len(X_test))

**Expected output**
```text
Training rows: 160
Test rows: 40
```

## 5. Train a baseline and a model
`DummyRegressor` provides a simple baseline that predicts the mean target from the training set. Linear regression is then compared against that baseline.

In [ ]:
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

model = LinearRegression()
model.fit(X_train, y_train)
model_predictions = model.predict(X_test)

print('Model coefficients:', model.coef_)
print('Model intercept:', round(model.intercept_, 2))

**Expected output:** two coefficients (one for each feature) and an intercept. Exact rounded values can vary slightly across library versions.

## 6. Evaluate with MAE and RMSE
MAE is the average absolute prediction error. RMSE is the square root of the mean squared error and penalizes larger errors more strongly. Both are expressed in the target's units.

In [ ]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {'MAE': mae, 'RMSE': rmse}

results = pd.DataFrame({
    'Baseline': regression_metrics(y_test, baseline_predictions),
    'Linear Regression': regression_metrics(y_test, model_predictions),
}).T
results.round(2)

**How to interpret:** lower MAE and RMSE indicate smaller errors on this test split. Since this is synthetic data generated with a linear relationship, linear regression is expected to perform better than the mean baseline. Always inspect the actual results rather than assuming this in advance.

## 7. Compare a few predictions
This comparison is illustrative; evaluation should use the full held-out test set, not just these rows.

In [ ]:
comparison = X_test.head(5).copy()
comparison['actual_score'] = y_test.head(5)
comparison['predicted_score'] = model_predictions[:5]
comparison.round(2)

## 8. Data leakage: a critical warning
Data leakage occurs when training or model-selection decisions use information that would not legitimately be available at prediction time. Examples include using a future outcome as a feature or fitting a scaler on the full dataset before splitting. Split first, and fit preprocessing only on training data.

## 9. Exercises
1. Change `test_size` to `0.25`. How many rows are in each split?
2. Change `RANDOM_STATE` and compare the metrics. Why can scores change?
3. Add a third feature that is unrelated to the target. Does the test score necessarily improve?
4. Explain why `score` must not be included in `X`.
5. Give a realistic example of data leakage in a prediction task.
6. Write two sentences comparing the baseline and linear regression results.

## Limitations
- The dataset is synthetic and intentionally simple.
- One train/test split does not fully characterize generalization.
- Model performance on this example should not be presented as evidence of performance on real-world data.

## Completion checklist
- [ ] Run every code cell from top to bottom.
- [ ] Answer all exercises in your own words.
- [ ] Explain features, target, split, baseline, MAE, RMSE, and leakage.
- [ ] Record one limitation and one next step.